# 🚀 PyTorch Core Fundamentals: An Interactive Study Guide
Welcome! This notebook is designed as a hands-on, conceptual, and practical deep dive into the **four essential pillars** of PyTorch:

1. **`torch`**: Tensors, mathematical operations, indexing, and the **Autograd** engine.
2. **`torch.nn`**: Neural network layers, activation functions, loss functions, and custom `nn.Module` classes.
3. **`torch.optim`**: Optimizers (SGD, Adam, AdamW), learning rates, and the canonical 5-step training loop.
4. **`torch.utils.data`**: Dataset abstraction (`Dataset`, `TensorDataset`) and batch streaming with `DataLoader`.

At the end of each module, you will find **Practice Exercises** to test your understanding. A **Capstone Challenge** at the end brings all four modules together in a real mini-project.
Solutions for all exercises are included at the bottom of the notebook!


---
## 0. Environment Setup & Device-Agnostic Code
In PyTorch, code should run seamlessly across CPUs, NVIDIA GPUs (CUDA), and Apple Silicon (MPS). We write device-agnostic code by querying available accelerators.


---
## Module 1: `torch` — The Tensor & Autograd Engine

The `torch` package is the core computational foundation of PyTorch. It provides multi-dimensional arrays called **Tensors** (similar to NumPy's `ndarray`), but with two massive superpowers:
1. **Hardware Acceleration**: Seamless execution on GPUs/TPUs.
2. **Autograd**: Automatic differentiation to compute gradients of arbitrary computational graphs.

### 1.1 Creating Tensors
Tensors can be initialized from Python lists, NumPy arrays, or built-in generator functions.


In [ ]:
# 1. From Python lists
a = torch.tensor([1.0, 2.0, 3.0])
print("From list:\n", a)

# 2. Common initialization functions
zeros = torch.zeros((2, 3))          # Shape (2, 3) filled with 0.0
ones = torch.ones((2, 3))            # Shape (2, 3) filled with 1.0
rand = torch.randn((2, 3))           # Samples from standard normal N(0, 1)
arange_tensor = torch.arange(0, 10, 2) # [0, 2, 4, 6, 8]
linspace_tensor = torch.linspace(0, 1, 5) # 5 evenly spaced values from 0 to 1

print("\nZeros:\n", zeros)
print("Standard Normal Random:\n", rand)
print("Arange:", arange_tensor)

# 3. Interoperability with NumPy (zero-copy memory sharing when possible)
np_arr = np.array([[1, 2], [3, 4]], dtype=np.float32)
torch_from_np = torch.from_numpy(np_arr)
back_to_np = torch_from_np.numpy()
print("\nPyTorch tensor from NumPy:\n", torch_from_np)


### 1.2 Tensor Attributes & Type Conversion
Every tensor has three primary metadata properties:
- **`shape`** (or `.size()`): The dimensions of the tensor.
- **`dtype`**: The data type (e.g., `torch.float32`, `torch.int64`, `torch.bool`).
- **`device`**: Where the tensor memory lives (`cpu` or `cuda:0`).


In [ ]:
t = torch.randn(3, 4, dtype=torch.float32)

print(f"Shape : {t.shape}")
print(f"Dtype : {t.dtype}")
print(f"Device: {t.device}")

# Type casting
t_int = t.to(torch.int32)        # Or t.int()
t_double = t.to(torch.float64)   # Or t.double()
print(f"Cast to int32 dtype: {t_int.dtype}")

# Moving to device
t_device = t.to(device)
print(f"Tensor on device: {t_device.device}")


### 1.3 Essential Tensor Operations: Math, Reshaping & Slicing
PyTorch supports element-wise math, matrix multiplications, reductions, and versatile dimension transformations.


In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# Element-wise operations
print("Addition (x + y):\n", x + y)
print("Element-wise multiplication (x * y):\n", x * y)

# Matrix Multiplication: torch.matmul or the '@' operator
# Shape: (2, 2) @ (2, 2) -> (2, 2)
matmul_res = x @ y
print("Matrix multiplication (x @ y):\n", matmul_res)

# Reductions
print("Sum of all elements:", x.sum().item())
print("Mean along columns (dim=0):", x.mean(dim=0))
print("Max along rows (dim=1):", x.max(dim=1).values)


#### Reshaping: `view`, `reshape`, `unsqueeze`, and `squeeze`
- **`view(*shape)`**: Reshapes tensor (requires contiguous memory).
- **`reshape(*shape)`**: Reshapes tensor (creates a copy if memory is non-contiguous).
- **`unsqueeze(dim)`**: Adds a dimension of size 1 at the specified index.
- **`squeeze(dim)`**: Removes dimensions of size 1.


In [ ]:
v = torch.arange(12)  # 1D tensor with 12 elements: [0, 1, ..., 11]
print("Original shape:", v.shape)

# Reshape into 3 rows, 4 columns
v_2d = v.view(3, 4)
print("Reshaped (3, 4):\n", v_2d)

# Unsqueeze: Add a batch dimension at dim=0 -> (1, 3, 4)
v_batched = v_2d.unsqueeze(0)
print("After unsqueeze(0) shape:", v_batched.shape)

# Squeeze: Remove dimension of size 1 -> (3, 4)
v_squeezed = v_batched.squeeze(0)
print("After squeeze(0) shape:", v_squeezed.shape)


### 1.4 The Autograd Engine: Automatic Differentiation
The magic of PyTorch is that it records operations on tensors with `requires_grad=True` to form a **Directed Acyclic Graph (DAG)** of computation.
When you call `.backward()` on a scalar loss, PyTorch computes gradients using the chain rule and stores them in `.grad`.

Key Autograd Rules:
1. Set `requires_grad=True` on tensors representing learnable parameters or inputs you want gradients for.
2. Call `loss.backward()` to populate `.grad`.
3. Wrap inference or evaluation code in `with torch.no_grad():` to disable graph tracking and save memory!


In [ ]:
# Define input tensor with autograd enabled
x = torch.tensor(3.0, requires_grad=True)

# Define a function: y = 2 * x^2 + 4 * x + 1
# Analytical derivative: dy/dx = 4 * x + 4
# At x = 3: dy/dx = 4(3) + 4 = 16
y = 2 * (x ** 2) + 4 * x + 1

print(f"x: {x}")
print(f"y: {y}")
print(f"y grad_fn: {y.grad_fn}")  # Shows the backward function node in computation graph

# Backpropagate to compute dy/dx
y.backward()

print(f"Computed gradient dy/dx at x=3: {x.grad.item()} (Expected: 16.0)")


#### Disabling Gradient Tracking (`torch.no_grad` and `.detach()`)
When evaluating a model or computing metrics, you do not want PyTorch to record operations in memory.


In [ ]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([5.0])

# With gradient tracking enabled:
out = w * x
print(f"out requires_grad: {out.requires_grad}")

# Inside torch.no_grad():
with torch.no_grad():
    out_eval = w * x
    print(f"Inside torch.no_grad, requires_grad: {out_eval.requires_grad}")

# Using .detach(): creates a detached copy sharing data without grad history
detached = out.detach()
print(f"Detached tensor requires_grad: {detached.requires_grad}")


### ✍️ Practice 1: Tensor Manipulation & Autograd Challenge

#### Task 1.1: Matrix Operations & Broadcasting
You have a batch of 5 samples, each with 3 features: `X` of shape `(5, 3)`.
You have weights `W` of shape `(3, 2)` and a bias `b` of shape `(2,)`.
Compute linear projection: $\hat{Y} = X W + b$.
Then, add a new singleton channel dimension so the output has shape `(5, 1, 2)`.

#### Task 1.2: Autograd Verification
Given the multivariable function:
$$f(u, v) = u^2 v + 3uv^3 - 7$$
1. Create tensors $u = 2.0$ and $v = -1.0$ with gradient tracking.
2. Compute $f(u, v)$.
3. Use `.backward()` to compute $\frac{\partial f}{\partial u}$ and $\frac{\partial f}{\partial v}$.
4. Print your results and compare with analytical formulas:
   - $\frac{\partial f}{\partial u} = 2uv + 3v^3$
   - $\frac{\partial f}{\partial v} = u^2 + 9uv^2$


In [ ]:
# --- PRACTICE 1 EXERCISE CODE CELL ---
# TODO: Complete Task 1.1 and 1.2 below

# Task 1.1:
X = torch.randn(5, 3)
W = torch.randn(3, 2)
b = torch.randn(2)

# Compute Y_hat = X @ W + b and unsqueeze to shape (5, 1, 2)
### YOUR CODE HERE ###
Y_hat = None

# Task 1.2:
# Define u and v with requires_grad=True
### YOUR CODE HERE ###
u = None
v = None
f = None

# Backpropagate
# f.backward()

# Verify your answer with the provided assertion check:
# (Uncomment after implementing)
# assert Y_hat.shape == (5, 1, 2), f"Expected shape (5, 1, 2), got {Y_hat.shape}"
# print("Task 1.1 PASSED!")
# print(f"Computed df/du: {u.grad.item():.4f}, df/dv: {v.grad.item():.4f}")


---
## Module 2: `torch.nn` — Neural Network Architectures

`torch.nn` contains all the core components needed to design, parameterize, and structure deep neural networks:
- **`nn.Module`**: The foundational base class for all neural network layers and entire models.
- **Layers**: Pre-built parameterized components (`nn.Linear`, `nn.Conv2d`, `nn.Dropout`, `nn.BatchNorm1d`, etc.).
- **Activation Functions**: Non-linear functions (`nn.ReLU`, `nn.Sigmoid`, `nn.GELU`, `nn.Softmax`).
- **Loss Functions (Criteria)**: Objective functions (`nn.MSELoss`, `nn.CrossEntropyLoss`, `nn.BCEWithLogitsLoss`).

### 2.1 Anatomy of `nn.Module`
When creating a custom model in PyTorch:
1. Subclass `nn.Module`.
2. Call `super().__init__()` in the constructor.
3. Define layers as member attributes (`self.layer_name = ...`).
4. Implement the `forward(self, x)` method describing how inputs pass through the network.


In [ ]:
class MultiLayerPerceptron(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int, num_classes: int):
        super().__init__()
        # Layer 1: Input to hidden
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.relu = nn.ReLU()
        
        # Regularization: Dropout randomly sets inputs to 0 with probability p during training
        self.dropout = nn.Dropout(p=0.2)
        
        # Layer 2: Hidden to output (logits)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Step 1: Linear projection -> activation
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Step 2: Final output layer (returns raw unnormalized logits)
        out = self.fc2(x)
        return out

# Instantiate model
model = MultiLayerPerceptron(in_features=8, hidden_dim=32, num_classes=3)
print(model)


### 2.2 Inspecting Model Parameters
When you define `nn.Linear` inside `nn.Module`, PyTorch automatically registers its weights and biases as learnable parameters.


In [ ]:
# Inspect all learnable parameters
for name, param in model.named_parameters():
    print(f"Layer: {name:15} | Shape: {str(param.shape):18} | Requires Grad: {param.requires_grad}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params}")


### 2.3 `nn.Sequential`: A Rapid Prototyping Shortcut
For simple architectures where data flows linearly from layer to layer, `nn.Sequential` provides a concise alternative to subclassing `nn.Module`.


In [ ]:
sequential_model = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 3)
)

dummy_input = torch.randn(4, 8) # Batch size 4, 8 input features
logits = sequential_model(dummy_input)
print("Sequential Model Output shape:", logits.shape)  # Expected: (4, 3)


### 2.4 Loss Functions: A Crucial Distinction
Choosing the correct loss function depends on your task:

| Task | Output Shape | Recommended Loss Function | Notes |
| :--- | :--- | :--- | :--- |
| **Regression** | `(N, 1)` or `(N, D)` | `nn.MSELoss()`, `nn.L1Loss()` | Compares continuous predictions to targets. |
| **Multi-class Classification** | `(N, C)` | `nn.CrossEntropyLoss()` | **Input MUST be raw logits** (do NOT apply Softmax before it!). |
| **Binary Classification** | `(N, 1)` or `(N)` | `nn.BCEWithLogitsLoss()` | **Input MUST be raw logits** (numerically more stable than Sigmoid + BCELoss). |

> ⚠️ **Common Beginner Pitfall**: `nn.CrossEntropyLoss` combines `nn.LogSoftmax()` and `nn.NLLLoss()` internally. If you apply a `nn.Softmax()` layer in your model and then use `nn.CrossEntropyLoss`, your model will receive double softmax and fail to train properly!


In [ ]:
# Example: Multi-class Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()

# Mock logits from model (batch_size=4, num_classes=3)
mock_logits = torch.randn(4, 3, requires_grad=True)
# Ground truth class indices (integer tensor: values 0, 1, or 2)
mock_targets = torch.tensor([0, 2, 1, 0], dtype=torch.long)

loss = criterion(mock_logits, mock_targets)
print(f"CrossEntropyLoss value: {loss.item():.4f}")


### ✍️ Practice 2: Building a Binary Classification Network

#### Task:
Construct a custom neural network named `BinaryClassifier` using `nn.Module` with the following specifications:
1. Input dimension: `10`
2. Hidden layer 1: `64` units followed by `nn.BatchNorm1d(64)` and `nn.ReLU()`
3. Hidden layer 2: `32` units followed by `nn.ReLU()` and `nn.Dropout(p=0.25)`
4. Output layer: `1` unit (producing raw scalar logits for binary classification)
5. Test it:
   - Create a dummy input batch `X_dummy = torch.randn(16, 10)`.
   - Perform a forward pass through your model.
   - Assert that the output shape is `(16, 1)`.
   - Compute `nn.BCEWithLogitsLoss` against mock labels `y_dummy = torch.randint(0, 2, (16, 1)).float()`.


In [ ]:
# --- PRACTICE 2 EXERCISE CODE CELL ---
# TODO: Implement BinaryClassifier subclassing nn.Module

class BinaryClassifier(nn.Module):
    def __init__(self, in_features: int = 10):
        super().__init__()
        ### YOUR CODE HERE ###
        pass
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ### YOUR CODE HERE ###
        pass

# Instantiate and test your model:
# model_binary = BinaryClassifier(in_features=10)
# X_dummy = torch.randn(16, 10)
# y_dummy = torch.randint(0, 2, (16, 1), dtype=torch.float32)

# out_dummy = model_binary(X_dummy)
# criterion_bce = nn.BCEWithLogitsLoss()
# loss_dummy = criterion_bce(out_dummy, y_dummy)

# print("Output shape:", out_dummy.shape)
# print("Dummy Loss:", loss_dummy.item())
# assert out_dummy.shape == (16, 1), f"Expected (16, 1), got {out_dummy.shape}"
# print("Practice 2 PASSED!")


---
## Module 3: `torch.optim` — Optimization Mechanics

`torch.optim` contains optimization algorithms that update model parameters according to the calculated gradients:
- **`optim.SGD`**: Stochastic Gradient Descent, with support for momentum and Nesterov acceleration.
- **`optim.Adam` / `optim.AdamW`**: Adaptive moment estimation. `AdamW` includes decoupled weight decay, the modern default choice for many deep learning architectures.

### 3.1 The Canonical 5-Step Training Loop
Every supervised PyTorch training iteration follows this exact sequence:

```python
# 1. Clear previous gradients
optimizer.zero_grad()

# 2. Forward pass: compute predictions
predictions = model(inputs)

# 3. Compute loss
loss = criterion(predictions, targets)

# 4. Backward pass: compute gradients via autograd
loss.backward()

# 5. Optimization step: update weights using gradients
optimizer.step()
```

#### Why `optimizer.zero_grad()` is MANDATORY:
By default, PyTorch **accumulates (sums)** gradients on every `.backward()` call.
If you omit `optimizer.zero_grad()`, gradients from the current batch will add onto gradients from previous batches, causing optimization to diverge or behave erratically.


In [ ]:
# Let's inspect a single training step manually on a simple linear model: y = 2x + 1
simple_model = nn.Linear(1, 1) # y = w * x + b
optimizer = optim.SGD(simple_model.parameters(), lr=0.1)
criterion = nn.MSELoss()

# Mock single data point: x = 2.0, target = 5.0 (since 2*2 + 1 = 5)
x_sample = torch.tensor([[2.0]])
y_target = torch.tensor([[5.0]])

print(f"Weight BEFORE step: {simple_model.weight.item():.4f}")
print(f"Bias BEFORE step  : {simple_model.bias.item():.4f}")

# Step 1: Zero out gradients
optimizer.zero_grad()

# Step 2: Forward pass
pred = simple_model(x_sample)

# Step 3: Compute loss
loss = criterion(pred, y_target)

# Step 4: Backward pass
loss.backward()
print(f"Computed weight gradient dLoss/dw: {simple_model.weight.grad.item():.4f}")

# Step 5: Optimizer step (w_new = w_old - lr * grad)
optimizer.step()

print(f"Weight AFTER step : {simple_model.weight.item():.4f}")
print(f"Bias AFTER step   : {simple_model.bias.item():.4f}")


### 3.2 Learning Rate Schedulers (`torch.optim.lr_scheduler`)
As training progresses, decreasing the learning rate often helps fine-tune weights closer to a minimum.


In [ ]:
# Example of StepLR: Decays lr by gamma every step_size epochs
optimizer = optim.AdamW(simple_model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print("Initial learning rate:", scheduler.get_last_lr()[0])
for epoch in range(1, 21):
    # (Inside training loop after optimizer.step())
    optimizer.step()
    scheduler.step()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} -> Updated LR: {scheduler.get_last_lr()[0]:.6f}")


### ✍️ Practice 3: Fit a Non-Linear Function with Gradient Descent

#### Task:
Train a small neural network to approximate the non-linear function:
$$y = \sin(x) + 0.1 \cdot \epsilon$$
over the interval $[-\pi, \pi]$.

1. Generate synthetic data:
   - $X = 200$ points evenly spaced using `torch.linspace(-torch.pi, torch.pi, 200).unsqueeze(1)`.
   - $y = \sin(X) + 0.05 \cdot \text{randn}$.
2. Build a model using `nn.Sequential` with:
   - `nn.Linear(1, 32)` -> `nn.Tanh()` -> `nn.Linear(32, 32)` -> `nn.Tanh()` -> `nn.Linear(32, 1)`.
3. Use `optim.Adam(model.parameters(), lr=0.01)` and `nn.MSELoss()`.
4. Run a training loop for 200 epochs implementing the **5 canonical steps**.
5. Log the loss every 50 epochs and verify the final loss is $< 0.05$.


In [ ]:
# --- PRACTICE 3 EXERCISE CODE CELL ---
# TODO: Implement the training loop to fit sin(x)

# 1. Create data
torch.manual_seed(42)
X_train = torch.linspace(-torch.pi, torch.pi, 200).unsqueeze(1)
y_train = torch.sin(X_train) + 0.05 * torch.randn_like(X_train)

# 2. Build model, optimizer, and loss criterion
### YOUR CODE HERE ###
sin_model = None
sin_optimizer = None
sin_criterion = None

# 3. Training Loop (200 epochs)
# for epoch in range(1, 201):
#     # Step 1: zero_grad
#     # Step 2: forward pass
#     # Step 3: loss
#     # Step 4: backward
#     # Step 5: step
#     pass

# Verify:
# with torch.no_grad():
#     final_loss = sin_criterion(sin_model(X_train), y_train).item()
# print(f"Final Sin Regression Loss: {final_loss:.5f}")
# assert final_loss < 0.05, f"Loss too high ({final_loss}), check architecture or training loop!"
# print("Practice 3 PASSED!")


---
## Module 4: `torch.utils.data` — Datasets & DataLoaders

Feeding entire datasets into memory or training with a single batch is rarely possible in practical machine learning.
PyTorch provides two distinct primitives:
1. **`Dataset`**: Represents an individual collection of samples and their labels. Maps an integer index `idx` to sample `(x, y)`.
2. **`DataLoader`**: An iterable engine that wraps a `Dataset` to provide:
   - Automatic batching (`batch_size`)
   - Shuffling (`shuffle=True`)
   - Multi-process data loading (`num_workers`)
   - Memory pinning (`pin_memory=True` for GPU acceleration)

### 4.1 In-Memory Fast Path: `TensorDataset`
When your entire dataset already fits into memory as PyTorch tensors, `TensorDataset` wraps them with zero boilerplate.


In [ ]:
features = torch.randn(100, 4)       # 100 samples, 4 features each
targets = torch.randint(0, 2, (100,)) # 100 binary labels

# Wrap in TensorDataset
tensor_ds = TensorDataset(features, targets)
print(f"Dataset length: {len(tensor_ds)}")

# Access a single element
sample_x, sample_y = tensor_ds[0]
print(f"Sample 0: Feature shape = {sample_x.shape}, Target = {sample_y.item()}")

# Wrap in a DataLoader
loader = DataLoader(tensor_ds, batch_size=16, shuffle=True)

# Iterate through one epoch
for batch_idx, (batch_features, batch_targets) in enumerate(loader):
    if batch_idx < 3: # Print first 3 batches
        print(f"Batch {batch_idx}: Features = {batch_features.shape}, Targets = {batch_targets.shape}")


### 4.2 Custom `Dataset`: Subclassing `torch.utils.data.Dataset`
For real-world workflows (images, audio, custom CSV files, or custom generative pipelines), you subclass `Dataset` and implement three methods:
1. `__init__(self, ...)`: Setup file paths, parameters, or data arrays.
2. `__len__(self)`: Return the total number of samples (`len(dataset)`).
3. `__getitem__(self, idx)`: Return the $i$-th sample `(feature, target)` as PyTorch tensors.


In [ ]:
class CustomTabularDataset(Dataset):
    def __init__(self, num_samples: int = 500):
        # Generate synthetic 3-class tabular data
        self.data = torch.randn(num_samples, 5)
        # Target label based on sign of linear combination
        self.targets = ((self.data[:, 0] + self.data[:, 1]) > 0).long()
        
    def __len__(self) -> int:
        return len(self.data)
        
    def __getitem__(self, idx: int):
        # Must return tensors for sample and label
        return self.data[idx], self.targets[idx]

# Instantiate custom dataset
custom_dataset = CustomTabularDataset(num_samples=250)
print(f"Total samples in dataset: {len(custom_dataset)}")

# Test random access
x_sample, y_sample = custom_dataset[42]
print(f"Sample #42: x={x_sample}, y={y_sample}")

# Load in batches of 32
custom_loader = DataLoader(custom_dataset, batch_size=32, shuffle=True, drop_last=False)
batch_x, batch_y = next(iter(custom_loader))
print(f"First batch shapes: x={batch_x.shape}, y={batch_y.shape}")


### ✍️ Practice 4: Custom Circular Coordinates Dataset

#### Task:
Create a custom `Dataset` named `CircleDataset` representing a 2D binary classification problem:
- Points $(x_1, x_2)$ are uniformly distributed in $[-1.5, 1.5]^2$.
- Label $y = 1$ if point falls inside radius $r \le 1.0$ ($x_1^2 + x_2^2 \le 1.0$), otherwise $y = 0$.

Requirements:
1. Subclass `torch.utils.data.Dataset`.
2. In `__init__`, take `num_samples: int` and generate 2D coordinates and binary float labels `(N, 1)`.
3. Implement `__len__` and `__getitem__`.
4. Wrap in a `DataLoader` with `batch_size=64` and `shuffle=True`.
5. Iterate over the DataLoader and verify the total number of batches.


In [ ]:
# --- PRACTICE 4 EXERCISE CODE CELL ---
# TODO: Implement CircleDataset subclassing Dataset

class CircleDataset(Dataset):
    def __init__(self, num_samples: int = 640):
        super().__init__()
        ### YOUR CODE HERE ###
        pass

    def __len__(self) -> int:
        ### YOUR CODE HERE ###
        pass

    def __getitem__(self, idx: int):
        ### YOUR CODE HERE ###
        pass

# Instantiate and verify:
# circle_ds = CircleDataset(num_samples=640)
# circle_loader = DataLoader(circle_ds, batch_size=64, shuffle=True)

# print(f"Dataset length: {len(circle_ds)}")
# print(f"DataLoader batch count: {len(circle_loader)}")

# first_batch_x, first_batch_y = next(iter(circle_loader))
# assert first_batch_x.shape == (64, 2), f"Expected (64, 2), got {first_batch_x.shape}"
# assert first_batch_y.shape == (64, 1), f"Expected (64, 1), got {first_batch_y.shape}"
# print("Practice 4 PASSED!")


---
## 🏆 Capstone Challenge: End-to-End Pipeline
Now let's bring all **4 core modules** together in harmony!

We will train a non-linear classifier to solve the non-linear concentric circle problem created in Practice 4:
1. **`torch.utils.data`**: Build training and validation `DataLoader`s from `CircleDataset`.
2. **`torch.nn`**: Design a non-linear Multi-Layer Perceptron classifier.
3. **`torch.optim`**: Setup `AdamW` optimizer and `BCEWithLogitsLoss`.
4. **`torch`**: Implement a full training & evaluation loop with:
   - Model switching between `model.train()` and `model.eval()`.
   - Gradient computation and weight updates.
   - Tracking validation loss and accuracy across epochs.

Target: Achieve **$\ge 95\%$ Validation Accuracy**!


In [ ]:
# --- CAPSTONE CHALLENGE CODE CELL ---
# 1. Prepare Datasets & Loaders (torch.utils.data)
train_dataset = CircleDataset(num_samples=1200) if 'CircleDataset' in globals() and CircleDataset(10).__len__() == 10 else None
# (If you haven't completed Practice 4 yet, see the Solutions section below first!)

# 2. Define Network Architecture (torch.nn)
class CircleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Raw logits for BCEWithLogitsLoss
        )
    def forward(self, x):
        return self.net(x)

# 3. Setup Training Components (torch.optim & torch.nn)
# model_capstone = CircleClassifier().to(device)
# optimizer_capstone = optim.AdamW(model_capstone.parameters(), lr=0.01)
# criterion_capstone = nn.BCEWithLogitsLoss()

# 4. Training Loop:
# Complete the 15-epoch training loop below:
# for epoch in range(1, 16):
#     model_capstone.train()
#     total_loss = 0.0
#     for batch_x, batch_y in train_loader:
#         batch_x, batch_y = batch_x.to(device), batch_y.to(device)
#         
#         # Step 1: zero_grad
#         # Step 2: forward pass
#         # Step 3: loss
#         # Step 4: backward
#         # Step 5: step
#         pass
#     print(f"Epoch {epoch:2d} | Avg Loss: {total_loss / len(train_loader):.4f}")


---
## 💡 Reference Solutions

Use these solutions to check your work or if you get stuck on any of the practice challenges.


### Solution: Practice 1


In [ ]:
# --- SOLUTION: PRACTICE 1 ---
# Task 1.1:
X = torch.randn(5, 3)
W = torch.randn(3, 2)
b = torch.randn(2)
# Projection and unsqueeze to add channel dimension
Y_hat = (X @ W + b).unsqueeze(1)
print("Task 1.1 Y_hat shape:", Y_hat.shape) # (5, 1, 2)

# Task 1.2:
u = torch.tensor(2.0, requires_grad=True)
v = torch.tensor(-1.0, requires_grad=True)
f = (u ** 2) * v + 3 * u * (v ** 3) - 7
f.backward()

# Analytical checks:
# df/du = 2*u*v + 3*v^3 = 2(2)(-1) + 3(-1) = -4 - 3 = -7
# df/dv = u^2 + 9*u*v^2 = (4) + 9(2)(1) = 4 + 18 = 22
print(f"Task 1.2 Autograd df/du: {u.grad.item():.1f} (Analytical: -7.0)")
print(f"Task 1.2 Autograd df/dv: {v.grad.item():.1f} (Analytical: 22.0)")


### Solution: Practice 2


In [ ]:
# --- SOLUTION: PRACTICE 2 ---
class BinaryClassifierSolution(nn.Module):
    def __init__(self, in_features: int = 10):
        super().__init__()
        self.layer1 = nn.Linear(in_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout = nn.Dropout(p=0.25)
        
        self.out_layer = nn.Linear(32, 1)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.relu1(self.bn1(self.layer1(x)))
        x = self.dropout(self.relu2(self.layer2(x)))
        return self.out_layer(x)

sol_model = BinaryClassifierSolution(in_features=10)
mock_x = torch.randn(16, 10)
mock_y = torch.randint(0, 2, (16, 1)).float()

logits = sol_model(mock_x)
bce_loss = nn.BCEWithLogitsLoss()(logits, mock_y)
print("Solution 2 Logits Shape:", logits.shape)
print("Solution 2 BCE Loss:", bce_loss.item())


### Solution: Practice 3


In [ ]:
# --- SOLUTION: PRACTICE 3 ---
torch.manual_seed(42)
X_train = torch.linspace(-torch.pi, torch.pi, 200).unsqueeze(1)
y_train = torch.sin(X_train) + 0.05 * torch.randn_like(X_train)

sin_model_sol = nn.Sequential(
    nn.Linear(1, 32),
    nn.Tanh(),
    nn.Linear(32, 32),
    nn.Tanh(),
    nn.Linear(32, 1)
)

sin_optimizer_sol = optim.Adam(sin_model_sol.parameters(), lr=0.01)
sin_criterion_sol = nn.MSELoss()

for epoch in range(1, 201):
    # 1. Zero grad
    sin_optimizer_sol.zero_grad()
    # 2. Forward
    preds = sin_model_sol(X_train)
    # 3. Loss
    loss = sin_criterion_sol(preds, y_train)
    # 4. Backward
    loss.backward()
    # 5. Step
    sin_optimizer_sol.step()
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | MSE Loss: {loss.item():.5f}")

print("Practice 3 Solution complete! Final Loss:", loss.item())


### Solution: Practice 4 & Complete Capstone


In [ ]:
# --- SOLUTION: PRACTICE 4 & CAPSTONE ---
class CircleDatasetSolution(Dataset):
    def __init__(self, num_samples: int = 1200):
        super().__init__()
        # Random points in [-1.5, 1.5]
        self.coords = (torch.rand(num_samples, 2) * 3.0) - 1.5
        # Radius squared = x1^2 + x2^2
        r_squared = self.coords[:, 0]**2 + self.coords[:, 1]**2
        # Target: 1 if inside unit circle, else 0
        self.labels = (r_squared <= 1.0).float().unsqueeze(1)
        
    def __len__(self) -> int:
        return len(self.coords)
        
    def __getitem__(self, idx: int):
        return self.coords[idx], self.labels[idx]

# 1. Create Datasets & DataLoaders
train_ds = CircleDatasetSolution(num_samples=1200)
val_ds = CircleDatasetSolution(num_samples=300)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

# 2. Model & Optimizer
capstone_model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

capstone_optim = optim.AdamW(capstone_model.parameters(), lr=0.01)
capstone_loss_fn = nn.BCEWithLogitsLoss()

# 3. Training Loop
print("Starting Capstone Training...")
for epoch in range(1, 16):
    capstone_model.train() # Set training mode (enables dropout/batchnorm updates)
    epoch_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        capstone_optim.zero_grad()
        logits = capstone_model(batch_x)
        loss = capstone_loss_fn(logits, batch_y)
        loss.backward()
        capstone_optim.step()
        
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    
    # 4. Evaluation Loop
    capstone_model.eval() # Set evaluation mode
    correct = 0
    total = 0
    with torch.no_grad(): # Disable autograd for evaluation
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = capstone_model(batch_x)
            # Convert logits to binary predictions using 0.0 threshold (sigmoid(0.0) = 0.5)
            preds = (logits >= 0.0).float()
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
            
    val_acc = (correct / total) * 100.0
    if epoch % 3 == 0 or epoch == 15:
        print(f"Epoch {epoch:2d}/15 | Train Loss: {avg_train_loss:.4f} | Val Accuracy: {val_acc:.1f}%")

print(f"\nCapstone Finished! Final Validation Accuracy: {val_acc:.2f}%")


---
## 🎯 Key Takeaways & Best Practices Checklist

1. **`torch` (Tensors & Autograd)**:
   - Tensors are multi-dimensional arrays with hardware acceleration and automatic gradient tracking.
   - Use `requires_grad=True` for parameters you want to optimize.
   - Always wrap validation/inference in `with torch.no_grad():` to save memory and compute.

2. **`torch.nn` (Modules & Layers)**:
   - Subclass `nn.Module` and define layers in `__init__`, routing data in `forward()`.
   - Prefer returning raw logits from your models and pairing with `nn.CrossEntropyLoss` or `nn.BCEWithLogitsLoss` for maximum numerical stability.
   - Switch modes with `model.train()` (training) and `model.eval()` (evaluation).

3. **`torch.optim` (Optimizers)**:
   - Remember the 5 canonical steps: `zero_grad()`, `forward()`, `loss()`, `backward()`, `step()`.
   - Don't forget `optimizer.zero_grad()` — PyTorch accumulates gradients by default!

4. **`torch.utils.data` (Data Pipelines)**:
   - Subclass `Dataset` implementing `__init__`, `__len__`, and `__getitem__`.
   - Wrap with `DataLoader` for seamless batching, shuffling, and worker streaming.
   - Keep datasets device-agnostic (transfer batches to `device` inside the training loop).

Happy Deep Learning with PyTorch! 🚀
